# 05 — GATU vs ViraLift: 5-record head-to-head

Five fully-annotated PRRSV records provide the **truth** (their own GenBank annotation).
Both tools receive the **same stripped, sequence-only FASTA** (`targets/*.fasta`) and the same
reference (`reference_PQ623173.gb`); neither sees the truth. Both are scored with the identical
metric used in `liftoff_compare.ipynb` and documented in `../VALIDATION_METHODOLOGY.md`:
**coordinate-only** (IoU >= 0.90 or +-6 bp, codon check OFF for both), **truth-anchored**,
denominator restricted to **reference genes present in truth (R n truth)**.

Records: `AF184212.1, AF325691.1, AY032626.1, AY262352.1, AY366525.1`. Reference `PQ623173.1`.

### Run order
1. **GATU** (manual, one at a time): load reference `reference_PQ623173.gb` + each
   `targets/<acc>.fasta`, accept the transferred annotations, and **save as GenBank** to
   `gatu_output/<accession>.gb` (e.g. `AF184212.1.gb`).
2. **Run this notebook** (needs `tblastn` on PATH). It lifts the 5 targets with ViraLift,
   ingests any GATU outputs present, and scores both.


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'app').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd, matplotlib.pyplot as plt
from Bio import SeqIO
from app.validation._shared.validation_utils import *
from app.src.lifting.tblastn_lifter import lift_all_tblastn
from app.src.alias.gene_alias import apply_alias_to_features
from app.src.pipeline import PipelineConfig

CASE = Path('outputs/gatu_5case')
PICK = ['AF184212.1','AF325691.1','AY032626.1','AY262352.1','AY366525.1']
REF  = CASE / 'reference_PQ623173.gb'
bundle = load_reference_bundle(REF)
CFG = PipelineConfig()   # production defaults, same as the accuracy harness
ref_names = sorted({f['name'] for f in bundle['features']})  # evaluable set R

truth_src = {r.id: r for r in load_genbank_records(DATA/'PRRS'/'PRRS_100seq_anno.gb') if r.id in PICK}
target_records = {p.stem: SeqIO.read(str(p),'fasta') for p in (CASE/'targets').glob('*.fasta')}
for acc in PICK:
    assert str(target_records[acc].seq) == str(truth_src[acc].seq), f'seq mismatch {acc}'
print('reference:', bundle['record'].id, '| R =', ref_names)
print('targets:', sorted(target_records))


In [ ]:
# Truth per record: R n truth (reference genes only), same as the accuracy harness
# dedupe_truth_by_name: one truth feature per name, matching how the comparator resolves
# truth (longest wins) and how the accuracy harness counts (record, gene) presence.
truth_by_acc = {acc: dedupe_truth_by_name(
                    parse_truth_features(truth_src[acc], bundle['alias_lookup'], bundle['feature_type'],
                                         filter_nested=False, target_names=ref_names, keep_extra_names=[])[0])
                for acc in PICK}

def coverage_rows(tool, acc, preds):
    """Truth-anchored, coordinate-only (codon OFF for both tools)."""
    truth = truth_by_acc[acc]
    if preds:
        cmp = compare_predictions_to_truth(preds, truth, codon_required_names=set())
        correct = set(cmp.loc[cmp['coord_correct'], 'pred_name']) if 'coord_correct' in cmp.columns else set()
    else:
        correct = set()
    return [{'tool': tool, 'accession': acc, 'gene': t['name'], 'found': t['name'] in correct} for t in truth]


In [ ]:
# ---- ViraLift (tblastn lift onto the SAME stripped target FASTA) ----
cov = []
for acc in PICK:
    # Same call shape as the accuracy harness (run_tblastn_against_truth): production
    # config passed explicitly, never the function signature's incidental defaults.
    lifted = lift_all_tblastn(
        ref_features=bundle['features'], ref_record=bundle['record'],
        query_record=target_records[acc],
        min_coverage=CFG.min_coverage, min_identity=CFG.min_identity,
        evalue=CFG.evalue, rescue_window=CFG.rescue_window,
        validate_codons=(bundle['feature_type']=='CDS'))
    preds = lifted_to_rows(acc, lifted, 'viralift')
    cov += coverage_rows('ViraLift', acc, preds)
print('ViraLift done:', sum(r['found'] for r in cov), '/', len(cov))


In [ ]:
# ---- GATU ingest (from outputs/gatu_5case/gatu_output/<acc>.gb) ----
def load_gatu_preds(acc):
    # GATU may drop the 2-letter accession prefix (AF184212.1 -> 184212.1);
    # match on filename suffix rather than exact prefix.
    hits = []
    for ext in ('gb','gbk','genbank','gbf'):
        hits += [p for p in (CASE/'gatu_output').glob(f'*.{ext}')
                 if acc.endswith(p.stem) or p.stem.endswith(acc)]
    if not hits:
        return None
    rec = load_single_genbank(hits[0])
    feats = parse_features_for_type(rec, 'CDS')
    if not feats:
        return []
    feats = apply_alias_to_features(feats, bundle['alias_lookup'])  # normalise names like truth
    return [{'record_id':acc,'method':'gatu','pred_name':f['name'],
             'pred_start':f['start'],'pred_end':f['end'],'strand':f['strand'],
             'has_start_codon':None,'has_stop_codon':None,'in_frame':None} for f in feats]

missing = []
for acc in PICK:
    preds = load_gatu_preds(acc)
    if preds is None:
        missing.append(acc); continue
    cov += coverage_rows('GATU', acc, preds)

if missing:
    print('GATU output NOT yet present for:', missing)
    print('-> run GATU, save to outputs/gatu_5case/gatu_output/<acc>.gb, then re-run this notebook.')
cov_df = pd.DataFrame(cov)
cov_df.to_csv(CASE/'coverage_per_gene.tsv', sep='\t', index=False)


In [ ]:
# ---- Summary (truth-anchored) ----
summary = (cov_df.groupby('tool').agg(truth_genes=('found','size'), correct=('found','sum')).reset_index())
summary['coord_pct'] = (summary['correct']/summary['truth_genes']*100).round(2)
summary.to_csv(CASE/'summary_5case.tsv', sep='\t', index=False)
print(summary.to_string(index=False))

# ORF5 line (primer-design target)
o = cov_df[cov_df['gene']=='ORF5']
print('\nORF5:', o.groupby('tool')['found'].agg(['sum','size']).to_dict('index'))


In [ ]:
# ---- Side-by-side per (accession, gene) + per-gene table ----
side = cov_df.pivot_table(index=['accession','gene'], columns='tool', values='found', aggfunc='first')
side.to_csv(CASE/'side_by_side.tsv', sep='\t')
per_gene = (cov_df.groupby(['gene','tool']).agg(n=('found','size'), correct=('found','sum')).reset_index())
per_gene['pct'] = (per_gene['correct']/per_gene['n']*100).round(1)
per_gene = per_gene.pivot_table(index='gene', columns='tool', values='pct', fill_value=0)
per_gene.to_csv(CASE/'per_gene.tsv', sep='\t')
per_gene


In [ ]:
# ---- Figure (drawn once GATU output is present) ----
if 'GATU' in set(cov_df['tool']):
    ax = summary.set_index('tool')['coord_pct'].reindex(['GATU','ViraLift']).plot(
        kind='bar', figsize=(5,4.2), color=['#b5651d','#2f7d4f'])
    ax.set_ylim(0,105); ax.set_ylabel('Coordinate correct (% of truth genes)'); ax.set_xlabel('')
    ax.set_title('GATU vs ViraLift (5 PRRSV records, R n truth, coordinate-only)')
    for c in ax.containers: ax.bar_label(c, fmt='%.1f', padding=3)
    ax.figure.tight_layout(); ax.figure.savefig(CASE/'gatu_vs_viralift.png', dpi=180, bbox_inches='tight')
else:
    print('Add GATU outputs to draw the comparison figure.')


In [ ]:
# ---- Per-gene IoU matrix: rows = (record x tool), cols = reference genes ----
# Each cell = IoU of that tool's lifted gene vs truth (1.0 = identical span).
#   blank = gene not annotated in that record's truth;  0.0 = not lifted / wrong region.
def iou_by_gene(acc, preds):
    truth = truth_by_acc[acc]
    if preds:
        cmp = compare_predictions_to_truth(preds, truth, codon_required_names=set())
        iou_map = dict(zip(cmp['pred_name'], cmp['iou'].round(3))) if 'iou' in cmp.columns else {}
    else:
        iou_map = {}
    return {t['name']: iou_map.get(t['name'], 0.0) for t in truth}

iou_long = []
for acc in PICK:
    v_lift = lift_all_tblastn(
        ref_features=bundle['features'], ref_record=bundle['record'],
        query_record=target_records[acc],
        min_coverage=CFG.min_coverage, min_identity=CFG.min_identity,
        evalue=CFG.evalue, rescue_window=CFG.rescue_window,
        validate_codons=(bundle['feature_type']=='CDS'))
    v_iou = iou_by_gene(acc, lifted_to_rows(acc, v_lift, 'viralift'))
    g_iou = iou_by_gene(acc, load_gatu_preds(acc) or [])
    for tool, d in (('ViraLift', v_iou), ('GATU', g_iou)):
        row = {'accession': acc, 'tool': tool}; row.update(d); iou_long.append(row)

iou_mat = pd.DataFrame(iou_long).set_index(['accession', 'tool'])
iou_mat = iou_mat.reindex(columns=[g for g in ref_names if g in iou_mat.columns])
iou_mat.to_csv(CASE / 'iou_matrix.tsv', sep='\t')
print('IoU per (record x tool) x gene  --  green=good lift, red=off, blank=not in truth')
iou_mat.style.background_gradient(cmap='RdYlGn', vmin=0, vmax=1, axis=None).format('{:.2f}', na_rep='')
